## 🚴‍♂️ Real-World Problem: Predicting Bicycle Rental Demand (subset selection - backward elimination)
 
 **Scenario: You work for a city bike rental company. You want to predict how many bikes will be rented on a given day.**

Data features we can consider:

|Feature	  |     Meaning                           |          
|-------------|---------------------------------------|
|temperature  |	  Average daily temperature (°C)
|humidity	  |   Humidity (%)
|windspeed	  |    Wind speed (km/h)
|day_of_week  |	  Monday=0 … Sunday=6
|holiday	  |    1 if it’s a public holiday, 0 otherwise
|working_day  |	  1 if it’s a working day, 0 otherwise
|season	      |    1=Spring, 2=Summer, 3=Fall, 4=Winter
|random_noise |	  A completely random variable, just for testing

**Target: rentals → number of bikes rented that day.**

### Step 1: Create Synthetic dataset

In [2]:
import pandas as pd
import numpy as np

n = 2000
np.random.seed(42)

#--------------------------
#numeric features
#--------------------------
temperature = np.random.randint(14, 30, n)
humidity = np.random.randint(0, 56, n)
windspeed = np.random.randint(0, 18, n)

#-----------------------------
# categorical(binary) features
#------------------------------
holiday = np.random.randint(0, 2, n)
working_day = np.random.randint(0, 2, n)

#--------------------------------------
# categorical(multiple labels) features
#--------------------------------------
season = np.random.randint(1, 5, n)
day_of_week = np.random.randint(0, 7, n)

# -----------------------------
# Noise feature
# -----------------------------
random_noise = np.random.randn(n)

# -----------------------------
# Create dataframe
# -----------------------------
df = pd.DataFrame({
    'temperature': temperature,
    'humidity': humidity,
    'windspeed': windspeed,
    
    'holiday': holiday,
    'working_day': working_day,
    
    'day_of_week': day_of_week,
    'season': season,
    
    'random_noise': random_noise
})

# ==============================================================================
# ONE-HOT ENCODING: SEASON AND DAY_OF_WEEK (using get_dummies function of pandas) 
# ==============================================================================

df = pd.get_dummies(
    df,
    columns=["season", "day_of_week"],
    drop_first=True   # VERY IMPORTANT (avoids multicollinearity)
)

# -----------------------------
# Target variable (rentals)
# -----------------------------
rentals = (

    200 * df["temperature"]
    -150 * df["humidity"]
    -100 * df["windspeed"]

    +500 * df["working_day"]
    +300 * df["holiday"]

    # season (spring is baseline, so NOT included)
    +600 * df["season_2"]
    +300 * df["season_3"]
    -500 * df["season_4"]

    # day_of_week (Monday is baseline)
    +50  * df["day_of_week_1"]
    +80  * df["day_of_week_2"]
    +60  * df["day_of_week_3"]
    +70  * df["day_of_week_4"]
    +100 * df["day_of_week_5"]
    +150 * df["day_of_week_6"]

    # noise
    + np.random.randn(n) * 100
)


# -----------------------------
# Add target
# -----------------------------
df["rentals"] = rentals

print(df.head())

   temperature  humidity  windspeed  holiday  working_day  random_noise  \
0           20        46         17        1            1      0.906544   
1           17        11          5        0            0      0.072252   
2           26        15         11        0            0      2.057811   
3           28        23         10        1            1     -1.438411   
4           24        18          1        0            0     -0.515171   

   season_2  season_3  season_4  day_of_week_1  day_of_week_2  day_of_week_3  \
0     False      True     False          False          False          False   
1     False     False     False          False          False          False   
2     False     False     False          False           True          False   
3     False     False     False          False          False          False   
4     False     False     False          False           True          False   

   day_of_week_4  day_of_week_5  day_of_week_6      rentals  
0     

### Step 2: Train and test sets

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns = "rentals")
y = df["rentals"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size= 0.2, random_state= 42)

### Step 3: Evaluation function

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def evaluate(features):
    # 1. Create linear regression model
    model = LinearRegression()
    
    # 2. Train only on selected features
    model.fit(X_train[features], y_train)
    
    # 3. Predict on test set
    preds = model.predict(X_test[features])
    
    # 4. Return MSE on test set
    return mean_squared_error(y_test, preds)

### Step 4: Backward Elimination

In [8]:
selected = list( X_train.columns)

prev_score = evaluate(selected)

print("Initial MSE:", prev_score)

while selected:

    scores = []

    # try removing each feature
    for f in selected:

        # temporary feature list
        temp_features = selected.copy()

        # remove one feature
        temp_features.remove(f)

        # evaluate remaining features
        score = evaluate(temp_features)

        scores.append((score, f))

    # sort by score (lower MSE is better)
    scores.sort()

    # best removal
    best_score, worst_feature = scores[0]

    # stopping condition
    if best_score - prev_score > 100:
        break

    # permanently remove feature
    selected.remove(worst_feature)

    prev_score = best_score

    print(f"Removed: {worst_feature}, MSE: {best_score:.2f}")

print("\nFinal selected features:")
print(selected)

Initial MSE: 8304.77361723227
Removed: random_noise, MSE: 8304.83

Final selected features:
['temperature', 'humidity', 'windspeed', 'holiday', 'working_day', 'season_2', 'season_3', 'season_4', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6']


----

## Backward Elimination 


### 1️⃣ What is Backward Elimination?

Backward Elimination is a **greedy feature selection method** where we start with **all features** and iteratively **remove the least useful feature** until removing any more **significantly worsens model performance**.

---

### 2️⃣ Why Use Backward Elimination?

- Reduce **model complexity**
- Remove **irrelevant or noisy features**
- Avoid **overfitting**
- Improve **interpretability**

---

### 3️⃣ Key Steps

1. Fit a model with **all features**.
2. Evaluate performance (e.g., **MSE** on test data).
3. For each feature:
   - Remove it temporarily.
   - Measure the change in MSE.
4. Identify the **least important feature** (removal changes MSE the least).
5. Permanently remove it.
6. Repeat until removing any feature **significantly increases error**.

---

### 4️⃣ Intuition

- Ask: *“Does the model still perform well without this feature?”*
  - **Yes → feature is not important → remove it**
  - **No → feature is important → keep it**
- **Minimal change in MSE** → feature likely unimportant
- **Large increase in MSE** → feature is important

---

### 5️⃣ Greedy Algorithm Concept

- **Greedy**: At each step, pick the feature whose removal has the **least negative impact**
- Fast but **not guaranteed** to find the absolute best subset

---

### 6️⃣ Handling Noise Features

- Random/noise features (e.g., `random_noise`) usually have **little effect** on MSE
- Backward elimination removes them first
- Prevents **overfitting** and keeps model meaningful

---

### 7️⃣ Multicollinearity Note

- Highly correlated features can mask each other’s importance
- Removing one may not change MSE much because the others “cover” it
- Helps identify **redundant features**

---

### 8️⃣ Interpretation of Selected Features

- Features that **remain at the end** are those that **truly contribute** to predicting the target
- Features **removed first** have **least predictive power** or are noisy
- Check magnitude and sign of coefficients to interpret **direction of influence**:
  - Positive coefficient → feature increases target
  - Negative coefficient → feature decreases target

---

### 9️⃣ Important Tips

- Always measure performance on **unseen data** (train-test split)
- Remove features **gradually**, do not remove multiple at once
- Stop when any further removal **worsens model performance**
- Works well for **moderate number of features**; computationally expensive for very large datasets

---

### ✅ Takeaway

Backward Elimination helps you:

1. **Simplify the model**
2. **Focus on important features**
3. **Improve generalization**
4. **Understand feature impact** via retained coefficients